# Inventory stocking: with MonteCarlo 

## Choose the best inventory stocking decision with respect to uncertain future demand. 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2022, University of Chicago 

---

We need to choose a weekly stocking level `X` in anticipation of uncertain weekly demand.

For each unit short for the week a cost of `C_s` is incurred.  For each unit stocked in excess of demand a cost of `C_e` is incurred.  

**Weekly demand follows a truncated Normal Distribution.**

What stocking decision `X` should be made to minimize the expected cost?

In [3]:
# === SETUP ===
import numpy as np
import os
from pprint import pprint
from pulp import *

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

np.random.seed(521)

In [2]:
C_s = 20
C_e = 10
old_demand = [ 230, 210, 160, 450, 310, 190, 130, 340, 370, 290 ]

### Create list of randomly generated demands

In [5]:
demand_arr = np.array(old_demand)
mean = np.mean(demand_arr)
std_dev = np.std(demand_arr)
print(mean)
print(std_dev)

268.0
96.20810776644555


In [6]:
num_scenarios = 1000

### Back to our old model code

In [ ]:
model = LpProblem("Inventory_Planning",LpMinimize)

### Define our First stage decision var

In [ ]:
X = pulp.LpVariable('X', lowBound=0, cat='Continuous')

### Define our Second stage decision vars

In [ ]:
S = []
E = []

for num in range(num_scenarios):
    S.append(pulp.LpVariable(f"S_{num}", lowBound=0, cat='Continuous'))
    E.append(pulp.LpVariable(f"E_{num}", lowBound=0, cat='Continuous'))

#print(S)
#print(E)

### Define the objective function

In [ ]:
obj = None
for num in range(num_scenarios):
    obj += 1.0/num_scenarios * ( C_e * E[num] + C_s * S[num]  )
model += obj, "objective function"

### Constraints involving Second Stage decision variables

In [ ]:
for num in range(num_scenarios):
    model += X + S[num] - E[num] == demand[num], f"InventoryBalance_{num}" 

In [ ]:
model.solve()
print(f"Status: {LpStatus[model.status]}")

In [ ]:
print(f"The objective function value is {value(model.objective)}")

In [ ]:
for v in model.variables():
    if (v.name=='X'):
        print(f"{v.name} = {v.varValue}")